In [1]:
from rdflib import Graph
from rdfine import GraphReader
from compilers import (
    PipelineGenerator,
    ProjectBuilder,
    LdioConfigCompiler)

#### Loading the graph

In [ ]:
# Loading the graph
input_folder = "..\\data\\"
catalog_graph = Graph()
graph_files = [
    "catalog-core.ttl",
    "catalog-ldio.ttl",
    "catalog-rdfc.ttl",
    "catalog-sw.ttl",
    "pipeline_definition.ttl",
    "catalog-application-profile-shapes.ttl",
]
for graph_file in graph_files:
    catalog_graph.parse(input_folder + graph_file, publicID = "file:///workspace/pipeline/")
catalog_reader = GraphReader(catalog_graph)
catalog_reader = catalog_reader.infer(input_folder + "inference_rules.yaml")

#### Validating the graph

    "Testing all shapes that are either part of the general application profile (defined in catalog-application-profile-shapes.ttl) or are attached to the pipeline components associated with the pipeline definition (defined in catalog-core.ttl / catalog-ldio.ttl / catalog-rdfc.ttl / catalog-sw.ttl)."

In [3]:
report = catalog_reader.validate(advanced=True, inference="rdfs")

if report.ask("?r sh:conforms true"):
    print("Graph conforms to all shapes.")
else:
    print("Violations:")
    print(report.select(
        "?focus ?message",
        "?r a sh:ValidationResult ; sh:focusNode ?focus ; sh:resultMessage ?message .",
    ).to_string(index=False))

Graph conforms to all shapes.


#### Compiling the pipeline build

`PipelineGenerator` orchestrates the full chain. It runs `PipelineExtractor` first (the only compiler that needs the `pipeline_id`, and it seeds the `tcs:PipelineBuild` node), then loops over `Compiler._registry`, invoking every compiler whose `applies_to` trigger becomes true against the growing build graph. Execution order emerges from those triggers rather than from any class-level rank.

In [4]:
pipeline_id = "demo:DishacledPipeline"
gen = PipelineGenerator(pipeline_id, catalog_reader.graph)
build_graph = gen.compile()

# Which compilers actually ran?
[cls.__name__ for cls in gen.compilers]

['PipelineExtractor',
 'PipelineAssembler',
 'LdioConfigCompiler',
 'RdfcConfigCompiler',
 'RdfcDockerFileCompiler',
 'SemanticWorksEnvVarCompiler',
 'VirtuosoCompiler',
 'MuClResourcesCompiler',
 'MuDispatcherCompiler',
 'MuDeltaNotifierCompiler',
 'MuAuthorizationCompiler',
 'ErrorAlertCompiler',
 'DockerComposeCompiler']

#### Inspecting the compiled files

    "Every compiler that produces a file attaches it to the `tcs:PipelineBuild` as an `spdx:File` node via `tcs:compiledFile`. The build graph is now self-describing: it knows which files should be written, where, and with what content.\n",

`ProjectBuilder` collects those nodes into a DataFrame on `builder.files` for inspection before any IO happens.

In [4]:
builder = ProjectBuilder(build_graph)

for _, row in builder.files.iterrows():
    print(f"=== {row['filepath']}/{row['filename']} ===")
    print(row['content'])
    print()

=== semantic-works/config/virtuoso/virtuoso.ini ===
;
;  virtuoso.ini
;
;  Configuration file for the OpenLink Virtuoso VDBMS Server
;
;  To learn more about this product, or any other product in our
;  portfolio, please check out our web site at:
;
;      http://virtuoso.openlinksw.com/
;
;  or contact us at:
;
;      general.information@openlinksw.com
;
;  If you have any technical questions, please contact our support
;  staff at:
;
;      technical.support@openlinksw.com
;

;
;  Database setup
;
[Database]
DatabaseFile			= /usr/local/virtuoso-opensource/var/lib/virtuoso/db/virtuoso.db
ErrorLogFile			= /usr/local/virtuoso-opensource/var/lib/virtuoso/db/virtuoso.log
LockFile			= /usr/local/virtuoso-opensource/var/lib/virtuoso/db/virtuoso.lck
TransactionFile			= /usr/local/virtuoso-opensource/var/lib/virtuoso/db/virtuoso.trx
xa_persistent_file		= /usr/local/virtuoso-opensource/var/lib/virtuoso/db/virtuoso.pxa
ErrorLogLevel			= 7
FileExtend			= 200
MaxCheckpointRemap		= 40000
Striping	

#### Writing the project to disk

`ProjectBuilder.write(target_dir)` materializes every collected file under the given directory, creating parent folders as needed. Existing files at the same path are overwritten. The call returns the absolute paths it wrote.

In [5]:
from pathlib import Path

full_out_dir = Path("../out/dishacled-full").resolve()
written = builder.write(str(full_out_dir))
for path in written:
    print(path)

C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\dishacled-full\semantic-works\config\virtuoso\virtuoso.ini
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\dishacled-full\semantic-works\config\error-alert\config.json
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\dishacled-full\docker-compose.yml
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\dishacled-full\semantic-works\config\resources\repository.lisp
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\dishacled-full\rdfc\Dockerfile
C:\Users\ThomasCarsten\OneDrive\Projects\Dishacled\github repo\toolchain-specification\pipeline generator\out\dishacled-full\semantic-works\config\authorization\config.lisp
C:\Users\ThomasCarsten\OneDri

#### Inspecting compiler internals

`PipelineGenerator` keeps the compiler instances it ran on `gen.compilers`, keyed by class. Each instance retains its intermediate state — useful for debugging when an output doesn't look right.

In [6]:
gen.compilers[LdioConfigCompiler].df_steps

,component,type,name,config
0,ldio:SparqlConstructTransformer,Transformer,Ldio:SparqlConstructTransformer,:config_2
1,ldio:RdfAdapter,Adapter,Ldio:RdfAdapter,NaN
2,ldio:HttpInPoller,Input,Ldio:HttpInPoller,:config_6
3,ldio:HttpOut,Output,Ldio:HttpOut,NaN


#### Inspecting what each compiler added and removed

Every compiler on `gen.compilers` exposes `.added_triples` and `.removed_triples` — `GraphReader` views over the delta between its `input_reader` (snapshot at construction time) and its `output_reader` (final state after `compile()`). Together they make the compilation process fully transparent: for any compiler, you can see exactly which triples it contributed and which it removed.

`SemanticWorksEnvVarCompiler` is a good example because it does both: it strips the old `tcs:literal` (or `tcs:embedded`) body of each SemanticWorks `tcs:DockerComposeConfig` and writes back an updated one with the step's config folded into the service `environment`.

In [7]:
from compilers import SemanticWorksEnvVarCompiler

sw = gen.compilers[SemanticWorksEnvVarCompiler]

print(f"Triples added by SemanticWorksEnvVarCompiler: {len(sw.added_triples.df)}")
print(f"Triples removed by SemanticWorksEnvVarCompiler: {len(sw.removed_triples.df)}")

print("\n--- added ---")
display(sw.added_triples.df)
print("\n--- removed ---")
display(sw.removed_triples.df)

Triples added by SemanticWorksEnvVarCompiler: 1
Triples removed by SemanticWorksEnvVarCompiler: 1

--- added ---


,sub,pred,obj,sub_type,obj_type
0,:LoketErrorAlertServiceDockerCompose,tcs:literal,"{""services"": {""error-alert"": {""image"": ""lblod/...",<class 'rdflib.term.URIRef'>,<class 'rdflib.term.Literal'>



--- removed ---


,sub,pred,obj,sub_type,obj_type
0,:LoketErrorAlertServiceDockerCompose,tcs:literal,\r\nerror-alert:\r\n image: lblod/loket-err...,<class 'rdflib.term.URIRef'>,<class 'rdflib.term.Literal'>
